# Lección 1 — Fundamentos de Chains

## 1.1 ¿Qué es un Chain?

Un **chain** es un pipeline donde el output de una llamada al LLM se transforma y se convierte en el input de la siguiente llamada.

En el ejercicio 2.4 hicimos chains manuales: extraer cambios → generar release notes → generar post de Ko-fi.

## 1.2 ¿Por qué no usar un solo prompt largo?

Imagina que le pides a Claude en un solo prompt: *"Lee estos commits, clasifícalos por tipo, genera release notes en markdown, tradúcelas al español, y también genera una versión corta para Play Store"*. Ese prompt sería enorme y Claude probablemente fallaría en alguna parte porque está haciendo demasiadas cosas a la vez.

Con un chain, cada paso tiene **UN objetivo claro**:

| Paso | Tarea | Output |
|------|-------|--------|
| 1 | Clasificar commits | `list[CommitEntry]` |
| 2 | Generar notas markdown | `str` (release notes) |
| 3 | Formatear para Play Store | `str` (max 500 chars) |
| 4 | Traducir al español | `str` (traducida) |

### La analogía de Unix pipelines

```bash
# En Unix, encadenas comandos simples:
cat access.log | grep "ERROR" | sort | uniq -c | head -10

# En un LLM chain, encadenas llamadas al LLM:
git_log | clasificar | generar_notas | formatear | traducir
```

Cada comando de Unix hace **una sola cosa bien**. Cada step de un chain hace lo mismo.

## 1.3 Anatomía de un Step

Cada step de un chain sigue esta estructura interna:

```
┌─────────────┐    ┌─────────────────┐    ┌──────────┐    ┌───────────────┐    ┌───────────┐
│ Input Parser│ →  │ Prompt Template │ →  │ LLM Call │ →  │ Output Parser │ →  │ Validator │
└─────────────┘    └─────────────────┘    └──────────┘    └───────────────┘    └───────────┘
```

- **Input Parser**: limpia y estructura los datos de entrada
- **Prompt Template**: construye el prompt con los datos
- **LLM Call**: llama a Claude con el prompt
- **Output Parser**: extrae la información de la respuesta (JSON, texto, etc.)
- **Validator**: valida el output (Pydantic, assertions, etc.)

Veámoslo con un ejemplo concreto — el clasificador de commits de Klimbook:

In [2]:
from typing import Literal
from pydantic import BaseModel

class CommitEntry(BaseModel):
    type: Literal["feature", "fix", "refactor", "docs", "chore"]
    description: str
    affected_service: str = "general"
    breaking: bool = False

In [ ]:
from anthropic import AsyncAnthropic
import json

client = AsyncAnthropic()

# 1. INPUT PARSER: recibe el raw git log y lo limpia
def parse_input(raw_git_log: str) -> list[str]:
    """Convierte el string de git log en una lista de commits."""
    lines = raw_git_log.strip().split("\n")
    return [line.strip() for line in lines if line.strip()]

# 2. PROMPT TEMPLATE: construye el prompt con los datos
def build_classify_prompt(commits: list[str]) -> str:
    """Genera el prompt para clasificar commits."""
    commits_text = "\n".join(commits)
    return f"""Classify each commit into a category.
<commits>
{commits_text}
</commits>

Return a JSON array with type, description, 
affected_service, and breaking for each."""

# 3. LLM CALL: llama a Claude
async def call_claude(prompt: str, model: str, temp: float):
    """Hace la llamada a la API."""
    response = await client.messages.create(
        model=model,
        max_tokens=2000,
        temperature=temp,
        messages=[
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": "["}  # prefill
        ]
    )
    return "[" + response.content[0].text

# 4. OUTPUT PARSER: extrae el JSON de la respuesta
def parse_output(raw_response: str) -> list[dict]:
    """Parsea el JSON de la respuesta de Claude."""
    return json.loads(raw_response)

# 5. VALIDATOR: valida con Pydantic
def validate(parsed: list[dict]) -> list[CommitEntry]:
    """Valida cada entry con Pydantic."""
    return [CommitEntry(**item) for item in parsed]

In [ ]:
# Step completo compuesto: parse → prompt → call → parse → validate
async def step_classify(raw_git_log: str) -> list[CommitEntry]:
    """Step completo: parse → prompt → call → parse → validate."""
    commits = parse_input(raw_git_log)
    prompt = build_classify_prompt(commits)
    raw_response = await call_claude(prompt, "claude-haiku-4-5-20251001", 0.0)
    parsed = parse_output(raw_response)
    entries = validate(parsed)
    return entries

### Gemma4

In [3]:
import json
from ollama import AsyncClient

client = AsyncClient()

# 1. INPUT PARSER: recibe el raw git log y lo limpia
def parse_input(raw_git_log: str) -> list[str]:
    """Convierte el string de git log en una lista de commits."""
    lines = raw_git_log.strip().split("\n")
    return [line.strip() for line in lines if line.strip()]

# 2. PROMPT TEMPLATE: construye el prompt con los datos
def build_classify_prompt(commits: list[str]) -> str:
    """Genera el prompt para clasificar commits."""
    commits_text = "\n".join(commits)
    return f"""Classify each commit into a category.
<commits>
{commits_text}
</commits>

Return a JSON array with type, description, 
affected_service, and breaking for each."""

# 3. LLM CALL: llama a Ollama
async def call_ollama(prompt: str, model: str, temp: float):
    """Hace la llamada a Ollama."""
    response = await client.chat(
        model=model,
        messages=[
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": "["}  # prefill
        ],
        options={
            "temperature": temp,
            "num_predict": 2000,
        },
    )
    return "[" + response["message"]["content"]

# 4. OUTPUT PARSER: extrae el JSON de la respuesta
def parse_output(raw_response: str) -> list[dict]:
    """Parsea el JSON de la respuesta de Ollama."""
    return json.loads(raw_response)

# 5. VALIDATOR: valida con Pydantic
def validate(parsed: list[dict]) -> list[CommitEntry]:
    """Valida cada entry con Pydantic."""
    return [CommitEntry(**item) for item in parsed]

In [4]:
# Step completo compuesto: parse → prompt → call → parse → validate
async def step_classify(raw_git_log: str) -> list[CommitEntry]:
    """Step completo: parse → prompt → call → parse → validate."""
    commits = parse_input(raw_git_log)
    prompt = build_classify_prompt(commits)
    raw_response = await call_ollama(prompt, "gemma3:12b", 0.0)
    parsed = parse_output(raw_response)
    entries = validate(parsed)
    return entries

## 1.4 State Management — Cómo pasar datos entre steps

Necesitas un lugar donde cada step lea sus inputs y escriba sus outputs. Tres opciones de menor a mayor formalidad:

In [5]:
def get_commits():
    """Función para obtener commits de ejemplo."""
    return "feat: add user authentication\nfix: correct typo in README\nrefactor: optimize database queries\ndocs: update API documentation" 

In [ ]:
# Opción 1: Diccionarios — simple, flexible, sin validación
state = {}
state["commits"] = get_commits()
state["classified"] = await step_classify(state["commits"])
state["notes"] = await step_generate(state["classified"])

# Opción 2: Dataclass — tipado pero sin validación automática
from dataclasses import dataclass, field

@dataclass
class PipelineState:
    raw_commits: list[str] = field(default_factory=list)
    classified: list[CommitEntry] = field(default_factory=list)
    notes_md: str = ""
    notes_playstore: str = ""
    notes_appstore: str = ""
    tokens_used: int = 0
    cost: float = 0.0

# Opción 3: Pydantic — tipado CON validación (recomendado)
class PipelineState(BaseModel):
    raw_commits: list[str] = []
    classified: list[CommitEntry] = []
    notes_md: str = ""
    notes_playstore: str = ""
    notes_appstore: str = ""
    tokens_used: int = 0
    cost: float = 0.0

## 1.5 Diseño de Pipelines

### Principio de responsabilidad única

Cada step debe hacer **UNA cosa**. Si te encuentras describiendo un step con "Y" en medio (*"clasifica commits Y genera notas"*), son dos steps.

In [ ]:
# MAL — un step que hace dos cosas:
async def step_classify_and_generate(commits):
    classified = classify(commits)
    notes = generate(classified)
    return notes

# BIEN — dos steps separados:
async def step_classify(commits) -> list[CommitEntry]:
    return classify(commits)

async def step_generate(classified) -> str:
    return generate(classified)

### Descomposición top-down

Empieza por el resultado final y trabaja hacia atrás preguntando *"¿qué necesito para esto?"*:

```
RESULTADO FINAL: Release notes en 5 formatos
    └── ¿Qué necesito? → notas en markdown base
        └── ¿Qué necesito? → commits clasificados por tipo
            └── ¿Qué necesito? → commits raw de git
                └── ¿Qué necesito? → dos tags de versión
```

Esto te da **4 steps naturales**: `git read → classify → generate → format`.

### Granularidad — el sweet spot

| Steps | Descripción |
|-------|-------------|
| **1 paso** | Prompt gigante, imposible de debuggear. Si falla no sabes dónde. |
| **2-5 pasos** | Cada paso manejable, fácil de debuggear, puedes reusar steps, costo razonable. |
| **10+ pasos** | Demasiado overhead de API calls, cada call tiene latencia (~1-3s), costo se multiplica, la complejidad explota. |

Para el Release Notes pipeline, **4-5 pasos** es perfecto:

1. **Git Reader** — (no LLM, solo código)
2. **Commit Classifier** — (Haiku, temp=0)
3. **Notes Generator** — (Sonnet, temp=0.3)
4. **Multi-formatter** — (Sonnet, paralelo)
5. **Validator** — (no LLM, solo código)

### Data flow — diagramar qué entra y qué sale de cada paso

```
Step 1: Git Reader
  Input:  version_from, version_to
  Output: list[str] — raw commit messages

Step 2: Classifier
  Input:  list[str] — raw commit messages
  Output: list[CommitEntry] — commits clasificados

Step 3: Generator
  Input:  list[CommitEntry] — commits clasificados
  Output: str — release notes en markdown

Step 4: Formatter
  Input:  str — release notes en markdown
  Output: dict[str, str] — {platform: formatted_notes}
```

> **Regla clave**: el output de cada paso debe coincidir con el input del siguiente. Si no coinciden, necesitas un paso intermedio de transformación.

### Testing individual

Cada paso debe ser testeable de forma aislada. Puedes crear inputs de prueba para cada step sin necesidad de correr todo el pipeline:

In [ ]:
# Test del clasificador aislado:
test_commits = [
    "abc123 feat(auth): add login endpoint",
    "def456 fix(db): correct connection string",
]
result = await step_classify(test_commits)
assert len(result) == 2
assert result[0].type == "feature"
assert result[1].type == "fix"

# Test del generador aislado:
test_classified = [
    CommitEntry(type="feature", description="add login", 
                affected_service="auth", breaking=False),
]
notes = await step_generate(test_classified)
assert "login" in notes

## 1.6 Manejo de Contexto entre Pasos

Este tema es crucial para producción porque afecta directamente tu costo.

### El problema del crecimiento cuadrático

Si cada paso recibe **TODO** el contexto de los pasos anteriores:

```
Step 1: envía   500 tokens de input → recibe  300 tokens
Step 2: envía   800 tokens de input → recibe  400 tokens  
Step 3: envía 2,000 tokens de input → recibe  500 tokens
Step 4: envía 4,500 tokens de input → ...
```

El costo crece exponencialmente. En un pipeline de 5 pasos, el último paso podría estar procesando **10x más tokens** que el primero.

### Selective forwarding — solo pasar lo necesario

In [ ]:
# MAL — pasa todo el contexto acumulado:
async def step_generate(state: PipelineState) -> str:
    prompt = f"""
    Raw commits: {state.raw_commits}      # ← innecesario
    Git metadata: {state.git_metadata}     # ← innecesario
    Classified: {state.classified}         # ← ESTO es lo único necesario
    Config: {state.config}                 # ← innecesario
    """

# BIEN — solo lo que necesita:
async def step_generate(classified: list[CommitEntry]) -> str:
    entries_text = "\n".join(
        f"- [{e.type}] {e.description}" for e in classified
    )
    prompt = f"""Generate release notes from these changes:
<changes>
{entries_text}
</changes>"""

La segunda versión envía ~200 tokens en vez de ~2,000. A $3/millón de tokens de input con Sonnet, la diferencia se acumula rápido en producción.

### Summarization step

Cuando el contexto crece demasiado, inserta un paso de resumen:

In [ ]:
# Imagina que tienes 200 commits (mucho texto)
# En vez de pasar los 200 commits al generador:

async def step_summarize(classified: list[CommitEntry]) -> str:
    """Resume los cambios clave antes de generar notas."""
    prompt = f"""Summarize these {len(classified)} changes 
into the 5-10 most important user-facing changes:
{json.dumps([e.model_dump() for e in classified])}"""
    
    return await call_claude(prompt, "claude-haiku-4-5-20251001", 0.0)

# Pipeline con summarization:
classified = await step_classify(commits)          # 200 entries
summary = await step_summarize(classified)          # 10 bullet points
notes = await step_generate_from_summary(summary)   # mucho menos input

### Gemma4

In [ ]:
# Versión Ollama del step de summarization
async def step_summarize(classified: list[CommitEntry]) -> str:
    """Resume los cambios clave antes de generar notas."""
    prompt = f"""Summarize these {len(classified)} changes 
into the 5-10 most important user-facing changes:
{json.dumps([e.model_dump() for e in classified])}"""
    
    return await call_ollama(prompt, "gemma3:12b", 0.0)

# Pipeline con summarization:
classified = await step_classify(commits)          # 200 entries
summary = await step_summarize(classified)          # 10 bullet points
notes = await step_generate_from_summary(summary)   # mucho menos input

### Context object pattern

Un objeto compartido donde cada paso lee y escribe **SOLO** los campos que le corresponden:

In [ ]:
class ChainContext(BaseModel):
    # Input
    version_from: str
    version_to: str
    
    # Step 1 output
    raw_commits: list[str] = []
    
    # Step 2 output
    classified: list[CommitEntry] = []
    
    # Step 3 output
    notes_md: str = ""
    
    # Step 4 output
    formatted: dict[str, str] = {}
    
    # Metadata
    total_tokens: int = 0
    total_cost: float = 0.0
    errors: list[str] = []

# Cada step recibe y modifica el contexto:
async def step_classify(ctx: ChainContext) -> ChainContext:
    ctx.classified = await classify(ctx.raw_commits)
    return ctx

async def step_generate(ctx: ChainContext) -> ChainContext:
    ctx.notes_md = await generate(ctx.classified)
    return ctx

## 1.7 Error Handling en Chains

En un solo prompt, si algo falla, simplemente reintentas. En un chain de 5 pasos, si el paso 4 falla después de que los pasos 1-3 costaron tokens, **no quieres repetir todo desde el inicio**.

### Fallas comunes y cómo manejarlas

In [ ]:
# 1. JSON inválido
try:
    parsed = json.loads(response)
except json.JSONDecodeError:
    # Claude devolvió texto en vez de JSON
    # → Retry con temperature=0 y prompt más explícito
    pass

# 2. Respuesta truncada
if response.stop_reason == "max_tokens":
    # Se cortó la respuesta
    # → Retry con max_tokens más alto
    # → O dividir el input en chunks más pequeños
    pass

# 3. Rate limit
# El SDK de Anthropic maneja esto automáticamente con retries
# Pero en parallel chains necesitas Semaphore

# 4. Respuesta irrelevante
if not any(keyword in response for keyword in expected_keywords):
    # Claude no entendió la tarea
    # → Retry con prompt más explícito
    pass

### Retry con escalation — cada intento es más estricto

In [ ]:
async def step_with_retry(input_data, max_retries=3):
    strategies = [
        {"temp": 0.3, "extra_instructions": ""},
        {"temp": 0.0, "extra_instructions": ""},
        {"temp": 0.0, "extra_instructions": 
            "You MUST respond with valid JSON only. "
            "No explanations, no markdown, just the JSON array."},
    ]
    
    for attempt, strategy in enumerate(strategies[:max_retries]):
        try:
            result = await call_and_parse(
                input_data, 
                temp=strategy["temp"],
                extra=strategy["extra_instructions"]
            )
            return result
        except (json.JSONDecodeError, ValidationError) as e:
            print(f"Intento {attempt+1} falló: {e}")
            if attempt == max_retries - 1:
                raise

### Validation gates — validar ANTES de pasar al siguiente paso

In [ ]:
async def run_pipeline(commits: list[str]):
    # Step 1: Classify
    classified = await step_classify(commits)
    
    # GATE 1: ¿Hay resultados?
    if not classified:
        raise PipelineError("La clasificación devolvió resultados vacíos")
    
    # GATE 2: ¿Todos tienen type válido?
    valid_types = {"feature", "fix", "refactor", "docs", "chore"}
    for entry in classified:
        if entry.type not in valid_types:
            raise PipelineError(f"Tipo inválido: {entry.type}")
    
    # Step 2: Generate (solo si pasó los gates)
    notes = await step_generate(classified)
    
    # GATE 3: ¿Las notas tienen contenido?
    if len(notes) < 50:
        raise PipelineError("Las notas generadas son demasiado cortas")
    
    # Step 3: Format
    formatted = await step_format(notes)
    
    # GATE 4: ¿Longitudes dentro de límites?
    if len(formatted["playstore"]) > 500:
        raise PipelineError("Las notas de Play Store exceden 500 caracteres")
    
    return formatted

### Fallback chains

Cuando el LLM falla repetidamente, ten un plan B que no dependa del LLM:

In [ ]:
async def step_classify(commits):
    try:
        # Intento principal: Claude clasifica
        return await classify_with_llm(commits)
    except MaxRetriesExceeded:
        # Fallback: clasificación simple por regex
        print("Clasificación LLM falló, usando fallback con regex")
        return classify_by_regex(commits)

def classify_by_regex(commits):
    """Clasificación básica sin LLM."""
    results = []
    for commit in commits:
        if commit.startswith("feat"):
            type_ = "feature"
        elif commit.startswith("fix"):
            type_ = "fix"
        elif commit.startswith("docs"):
            type_ = "docs"
        else:
            type_ = "chore"
        results.append(CommitEntry(
            type=type_, 
            description=commit, 
            affected_service="general"
        ))
    return results

### Logging estructurado — registrar TODO

En producción, sin logs no puedes debuggear. Registra input, output, tokens, tiempo y errores de cada paso:

In [ ]:
import logging
import time

logger = logging.getLogger("pipeline")

async def step_classify(commits: list[str]) -> list[CommitEntry]:
    step_name = "classify"
    logger.info(f"[{step_name}] Iniciando | Input: {len(commits)} commits")
    
    start = time.time()
    tokens_in = 0
    tokens_out = 0
    
    try:
        result = await call_classify(commits)
        elapsed = time.time() - start
        
        logger.info(
            f"[{step_name}] Éxito | "
            f"Output: {len(result)} entries | "
            f"Tokens: in={tokens_in} out={tokens_out} | "
            f"Tiempo: {elapsed:.2f}s | "
            f"Costo: ${calculate_cost(tokens_in, tokens_out):.4f}"
        )
        return result
        
    except Exception as e:
        elapsed = time.time() - start
        logger.error(
            f"[{step_name}] Falló | "
            f"Error: {type(e).__name__}: {e} | "
            f"Tiempo: {elapsed:.2f}s"
        )
        raise

Cuando debuggeas un pipeline, el log debería verse así:

```
[classify]  Iniciando | Input: 15 commits
[classify]  Éxito     | Output: 15 entries | Tokens: in=890 out=420 | Tiempo: 1.23s | Costo: $0.0012
[generate]  Iniciando | Input: 15 entries
[generate]  Éxito     | Output: 1 doc | Tokens: in=340 out=580 | Tiempo: 2.15s | Costo: $0.0028
[format]    Iniciando | Input: 1 doc
[format]    Falló     | Error: JSONDecodeError: Expecting value | Tiempo: 1.87s
[format]    Retry 1   | temp=0.0
[format]    Éxito     | Output: 5 formats | Tokens: in=340 out=890 | Tiempo: 2.34s | Costo: $0.0038
[PIPELINE]  Completo  | Total tokens: 3060 | Total costo: $0.0078 | Total tiempo: 7.59s
```

## 1.8 Ejecución Paralela en Chains

No todos los steps de un chain son secuenciales. Si dos pasos no dependen uno del otro, pueden correr en paralelo con `asyncio.gather`:

```
          ┌── format_playstore() ──┐
          │                        │
generate ─┼── format_appstore()  ──┼─ combinar resultados
          │                        │
          └── format_kofi()      ──┘
```

Esto reduce la latencia total: en vez de esperar 3 calls secuenciales (~6s), esperas solo la más lenta (~2s).

In [ ]:
import asyncio

# SECUENCIAL — cada format espera al anterior (lento):
async def step_format_sequential(notes: str) -> dict[str, str]:
    playstore = await format_for_playstore(notes)   # ~2s
    appstore = await format_for_appstore(notes)      # ~2s
    kofi = await format_for_kofi(notes)              # ~2s
    return {"playstore": playstore, "appstore": appstore, "kofi": kofi}
    # Total: ~6s

# PARALELO — todos corren a la vez (rápido):
async def step_format_parallel(notes: str) -> dict[str, str]:
    playstore, appstore, kofi = await asyncio.gather(
        format_for_playstore(notes),   # ─┐
        format_for_appstore(notes),    # ─┼─ corren simultáneamente
        format_for_kofi(notes),        # ─┘
    )
    return {"playstore": playstore, "appstore": appstore, "kofi": kofi}
    # Total: ~2s (la más lenta)

# PARALELO CON CONTROL — limitar concurrencia para no saturar la API:
async def step_format_controlled(notes: str, max_concurrent: int = 3):
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def limited_call(coro):
        async with semaphore:
            return await coro
    
    results = await asyncio.gather(
        limited_call(format_for_playstore(notes)),
        limited_call(format_for_appstore(notes)),
        limited_call(format_for_kofi(notes)),
    )
    return dict(zip(["playstore", "appstore", "kofi"], results))

## 1.9 Checkpointing — Guardar resultados intermedios

En un pipeline de 5 pasos, si el paso 4 falla, no quieres repetir los pasos 1-3 que ya costaron tokens y tiempo. La solución es guardar el resultado de cada paso:

### Caché simple con archivos

In [ ]:
import json
from pathlib import Path

CACHE_DIR = Path(".pipeline_cache")
CACHE_DIR.mkdir(exist_ok=True)

def save_checkpoint(step_name: str, data, run_id: str):
    """Guarda el output de un step en disco."""
    path = CACHE_DIR / f"{run_id}_{step_name}.json"
    path.write_text(json.dumps(data, default=str))

def load_checkpoint(step_name: str, run_id: str):
    """Carga un checkpoint previo si existe."""
    path = CACHE_DIR / f"{run_id}_{step_name}.json"
    if path.exists():
        return json.loads(path.read_text())
    return None

# Uso en el pipeline:
async def run_pipeline(commits, run_id="release_v2.9"):
    # Step 1: Classify (usa caché si existe)
    cached = load_checkpoint("classify", run_id)
    if cached:
        print("Usando clasificación cacheada")
        classified = [CommitEntry(**c) for c in cached]
    else:
        classified = await step_classify(commits)
        save_checkpoint("classify", [c.model_dump() for c in classified], run_id)
    
    # Step 2: Generate
    cached = load_checkpoint("generate", run_id)
    if cached:
        print("Usando notas cacheadas")
        notes = cached
    else:
        notes = await step_generate(classified)
        save_checkpoint("generate", notes, run_id)
    
    return notes

## 1.10 ¿Cuándo NO usar chains?

No todo necesita un chain. Usa un **solo prompt** cuando:

- La tarea es simple y autocontenida (ej: resumir un texto corto)
- No necesitas validación intermedia
- El output no alimenta otra llamada al LLM
- La latencia es crítica y no puedes permitir múltiples round-trips

Usa un **chain** cuando:

- La tarea tiene múltiples subtareas con diferentes requisitos (modelos, temperaturas)
- Necesitas validar outputs intermedios antes de continuar
- El output de un paso requiere transformación antes del siguiente
- Quieres reusar steps individuales en diferentes pipelines
- Necesitas fallbacks granulares (no reintentar todo si solo un paso falla)

> **Regla general**: si puedes describir la tarea en una sola oración sin usar "Y" o "luego", probablemente no necesitas un chain.

## 1.11 Python Puro vs Frameworks

### Python puro (lo que usamos)

```python
async def release_notes_pipeline(version_from, version_to):
    commits = read_git_log(version_from, version_to)    # no LLM
    classified = await step_classify(commits)            # Haiku
    notes = await step_generate(classified)              # Sonnet
    formatted = await step_format(notes)                 # Sonnet, paralelo
    return formatted
```

**Pros**: máximo control, sabes exactamente qué pasa, fácil de debuggear, sin dependencias extra.

### LangChain (referencia)

```python
from langchain import LLMChain, PromptTemplate
from langchain_anthropic import ChatAnthropic

# LCEL (LangChain Expression Language):
chain = classify_prompt | model | output_parser
```

**Pros**: ecosistema grande, integraciones pre-hechas.
**Contras**: abstracción excesiva, API cambia frecuentemente, difícil debuggear cuando algo falla dentro de la abstracción.

### LlamaIndex (referencia)

Más enfocado en RAG y datos. Si tu chain involucra mucha búsqueda en documentos, LlamaIndex tiene buenas abstracciones para eso.

### La postura

> Construye con **Python puro**. Cuando entiendas perfectamente los primitivos (prompts, API calls, parsing, validation, retry), podrás evaluar si un framework te aporta algo o solo agrega complejidad innecesaria.

---

## Resumen de Conceptos Clave

| Concepto | Descripción |
|----------|-------------|
| **Chain** | Pipeline donde output de un LLM call → input del siguiente |
| **Anatomía de un step** | Input parser → prompt template → LLM call → output parser → validator |
| **Responsabilidad única** | Cada paso hace UNA cosa |
| **Selective forwarding** | Solo pasar los datos que el siguiente paso necesita |
| **Validation gates** | Validar entre pasos antes de continuar |
| **Retry con escalation** | temp normal → temp=0 → prompt más explícito → fallback |
| **Ejecución paralela** | `asyncio.gather` para steps independientes |
| **Checkpointing** | Guardar resultados intermedios para no repetir pasos costosos |
| **Logging** | Registrar input, output, tokens, tiempo y errores de cada paso |
| **Python puro** | Máximo control, después puedes evaluar frameworks |